In [31]:
import os
import math
import random
from itertools import combinations

import numpy as np
import pandas as pd
import tensorflow as tf

In [32]:
# =========================================================
# 基本设置
# =========================================================
# 做诊断实验，CPU 更稳；想用 GPU 就改成 "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

SEED = 8
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 1024

BASE_FILES = {
    2.4: "./24Train.csv",
    2.5: "./25Train.csv",
    2.6: "./26Train.csv",
}

FEATURE_COLS = ["freq", "W1", "W2", "W3", "W4", "W5", "W6", "W7", "W8", "L1", "L2", "L3", "L4"]
TARGET_COLS = ["S11r", "S11i", "S21r", "S21i", "S31r", "S31i", "S41r", "S41i"]

# output distance 用几何参数对齐
GEOM_COLS = [c for c in FEATURE_COLS if c != "freq"]

In [33]:
# low / mid / high 的客户端混合比例
# 注意：这里只是决定“客户端拿到哪些样本”
# 样本本身的 freq 不改，保持输入输出自洽
SCENARIO_CONFIGS = {
    "low": {
        2.4: {2.4: 1/3, 2.5: 1/3, 2.6: 1/3},
        2.5: {2.4: 1/3, 2.5: 1/3, 2.6: 1/3},
        2.6: {2.4: 1/3, 2.5: 1/3, 2.6: 1/3},
    },
    "mid": {
        2.4: {2.4: 0.70, 2.5: 0.15, 2.6: 0.15},
        2.5: {2.4: 0.15, 2.5: 0.70, 2.6: 0.15},
        2.6: {2.4: 0.15, 2.5: 0.15, 2.6: 0.70},
    },
    "high": {
        2.4: {2.4: 1.00, 2.5: 0.00, 2.6: 0.00},
        2.5: {2.4: 0.00, 2.5: 1.00, 2.6: 0.00},
        2.6: {2.4: 0.00, 2.5: 0.00, 2.6: 1.00},
    },
}

In [34]:
# =========================================================
# 你的原始 MLP
# =========================================================
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [35]:
# =========================================================
# 数据读取
# =========================================================
def load_base_datasets(base_files):
    dfs = {}
    for freq, path in base_files.items():
        df = pd.read_csv(path, encoding="utf-8").copy()
        dfs[freq] = df.reset_index(drop=True)
        print(f"Loaded {freq} GHz: {len(df)} samples from {path}")
    return dfs

In [36]:
# =========================================================
# 构造 low / mid / high 客户端数据
# 关键：保持样本原始 freq，不做篡改
# =========================================================
def sample_mixed_client_dataset_keep_consistent(
    source_dfs,
    mix_cfg,
    total_size=None,
    seed=42,
):
    rng = np.random.default_rng(seed)

    if total_size is None:
        total_size = min(len(df) for df in source_dfs.values())

    src_freqs = sorted(source_dfs.keys())
    counts = {}
    remain = total_size

    for i, f in enumerate(src_freqs):
        if i < len(src_freqs) - 1:
            c = int(round(total_size * mix_cfg.get(f, 0.0)))
            counts[f] = c
            remain -= c
        else:
            counts[f] = remain

    parts = []
    for f in src_freqs:
        c = counts[f]
        if c <= 0:
            continue

        src_df = source_dfs[f]
        replace = c > len(src_df)
        idx = rng.choice(len(src_df), size=c, replace=replace)

        # 关键：不改 freq，保持输入-标签自洽
        part = src_df.iloc[idx].copy()
        parts.append(part)

    out = pd.concat(parts, axis=0, ignore_index=True)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out


def build_scenario_client_datasets(base_dfs, scenario_name, total_size=None, seed=42):
    """
    返回该场景下三个客户端的本地数据
    用于 gradient divergence
    """
    cfg = SCENARIO_CONFIGS[scenario_name]
    client_dfs = {}

    for client_freq in sorted(base_dfs.keys()):
        client_dfs[client_freq] = sample_mixed_client_dataset_keep_consistent(
            source_dfs=base_dfs,
            mix_cfg=cfg[client_freq],
            total_size=total_size,
            seed=seed + int(client_freq * 100),
        )

    return client_dfs

In [37]:
# =========================================================
# Output Distance
# =========================================================
def compute_pairwise_output_distance(df_a, df_b, geom_cols, target_cols):
    """
    按几何参数对齐，计算两个频率下同一几何的标签距离
    这里应该用原始纯净数据 base_dfs，而不是混合后的 client_dfs
    """
    a = df_a[geom_cols + target_cols].copy()
    b = df_b[geom_cols + target_cols].copy()

    a = a.rename(columns={c: f"{c}_a" for c in target_cols})
    b = b.rename(columns={c: f"{c}_b" for c in target_cols})

    merged = pd.merge(a, b, on=geom_cols, how="inner")

    if len(merged) == 0:
        raise ValueError("按几何参数 merge 后没有公共样本，请检查 GEOM_COLS 是否正确。")

    ya = merged[[f"{c}_a" for c in target_cols]].to_numpy(dtype=np.float32)
    yb = merged[[f"{c}_b" for c in target_cols]].to_numpy(dtype=np.float32)

    dists = np.linalg.norm(ya - yb, axis=1)
    return {
        "num_matched": int(len(merged)),
        "output_distance": float(np.mean(dists)),
        "output_distance_std": float(np.std(dists)),
        "output_distance_median": float(np.median(dists)),
    }


def compute_output_distance_for_scenario(base_dfs, scenario_name, geom_cols, target_cols):
    """
    准确版 output distance:
    - low: 三个客户端频率分布几乎一样，所以客户端间 output distance 应接近 0
    - mid/high: 根据客户端频率分布的差异，对原始 pairwise distance 做期望加权

    先计算原始三频之间的 pairwise response distance：
        D_24_25, D_24_26, D_25_26
    再根据场景中各客户端的频率混合比例，算期望 output distance
    """
    freqs = sorted(base_dfs.keys())
    raw_dist = {}

    # 原始频率对之间的真实距离
    for fa, fb in combinations(freqs, 2):
        stats = compute_pairwise_output_distance(base_dfs[fa], base_dfs[fb], geom_cols, target_cols)
        raw_dist[(fa, fb)] = stats

    cfg = SCENARIO_CONFIGS[scenario_name]

    pair_rows = []
    weighted_means = []

    for ca, cb in combinations(freqs, 2):
        mix_a = cfg[ca]
        mix_b = cfg[cb]

        # E[distance] = sum_p sum_q pi_a(p) pi_b(q) D(p,q)
        # 其中 D(p,p)=0, D(p,q)=raw pairwise distance
        exp_mean = 0.0
        exp_second = 0.0

        for p in freqs:
            for q in freqs:
                w = mix_a.get(p, 0.0) * mix_b.get(q, 0.0)
                if p == q:
                    d = 0.0
                else:
                    key = tuple(sorted((p, q)))
                    d = raw_dist[key]["output_distance"]
                exp_mean += w * d
                exp_second += w * (d ** 2)

        exp_std = max(exp_second - exp_mean ** 2, 0.0) ** 0.5

        pair_rows.append({
            "scenario": scenario_name,
            "client_pair": f"{ca} vs {cb}",
            "client_a": ca,
            "client_b": cb,
            "output_distance": float(exp_mean),
            "output_distance_std": float(exp_std),
        })
        weighted_means.append(exp_mean)

    summary = {
        "pairwise": pd.DataFrame(pair_rows),
        "overall_mean_pairwise_l2": float(np.mean(weighted_means)),
        "raw_pairwise_base": pd.DataFrame([
            {
                "client_pair": f"{k[0]} vs {k[1]}",
                "freq_a": k[0],
                "freq_b": k[1],
                "num_matched": v["num_matched"],
                "output_distance": v["output_distance"],
                "output_distance_std": v["output_distance_std"],
                "output_distance_median": v["output_distance_median"],
            }
            for k, v in raw_dist.items()
        ])
    }
    return summary

In [38]:
# =========================================================
# Gradient Divergence
# =========================================================
def get_fixed_batch(df, feature_cols, target_cols, batch_size=64, seed=42):
    rng = np.random.default_rng(seed)
    n = len(df)
    size = min(batch_size, n)
    idx = rng.choice(n, size=size, replace=False)

    x = df.iloc[idx][feature_cols].to_numpy(dtype=np.float32)
    y = df.iloc[idx][target_cols].to_numpy(dtype=np.float32)
    return tf.convert_to_tensor(x), tf.convert_to_tensor(y)


def grads_to_numpy_list(grads):
    out = []
    for g in grads:
        if g is None:
            continue
        out.append(g.numpy().astype(np.float32, copy=False))
    return out


def grad_sq_norm(grad_list):
    s = 0.0
    for g in grad_list:
        s += np.sum(g * g, dtype=np.float64)
    return float(s)


def grad_dot(grad_list_a, grad_list_b):
    s = 0.0
    for ga, gb in zip(grad_list_a, grad_list_b):
        s += np.sum(ga * gb, dtype=np.float64)
    return float(s)


def compute_client_gradient_stats(model, x, y):
    with tf.GradientTape() as tape:
        y_pred = model(x, training=False)
        loss = tf.reduce_mean(tf.square(y_pred - y))

    grads = tape.gradient(loss, model.trainable_variables)
    grads_np = grads_to_numpy_list(grads)
    sq_norm = grad_sq_norm(grads_np)

    return grads_np, sq_norm, float(loss.numpy())


def compute_gradient_divergence_summary(
    client_dfs,
    feature_cols,
    target_cols,
    batch_size=64,
    seed=42,
):
    """
    准确版 gradient divergence:
    - 基于 low/mid/high 混合后的客户端数据
    - 保持输入/标签自洽
    - 所有客户端在同一个初始化模型上算梯度
    """
    model = MLP()
    dummy_x = tf.zeros((1, len(feature_cols)), dtype=tf.float32)
    model(dummy_x)

    grad_dict = {}
    sqnorm_dict = {}
    loss_dict = {}

    for i, client_id in enumerate(sorted(client_dfs.keys())):
        x, y = get_fixed_batch(
            client_dfs[client_id],
            feature_cols,
            target_cols,
            batch_size=batch_size,
            seed=seed + i * 100,
        )

        grads_np, sq_norm, loss = compute_client_gradient_stats(model, x, y)
        grad_dict[client_id] = grads_np
        sqnorm_dict[client_id] = sq_norm
        loss_dict[client_id] = loss

    clients = sorted(grad_dict.keys())

    # pairwise gradient distance
    pair_rows = []
    pair_dists = []

    for ca, cb in combinations(clients, 2):
        ga = grad_dict[ca]
        gb = grad_dict[cb]

        dot_ab = grad_dot(ga, gb)
        sq_a = sqnorm_dict[ca]
        sq_b = sqnorm_dict[cb]

        dist_sq = max(sq_a + sq_b - 2.0 * dot_ab, 0.0)
        dist = math.sqrt(dist_sq)
        cos = dot_ab / (math.sqrt(sq_a) * math.sqrt(sq_b) + 1e-12)

        pair_rows.append({
            "client_pair": f"{ca} vs {cb}",
            "client_a": ca,
            "client_b": cb,
            "gradient_distance": float(dist),
            "gradient_cosine": float(cos),
        })
        pair_dists.append(dist)

    # divergence to mean gradient
    mean_grad = []
    num_clients = len(clients)

    for layer_idx in range(len(grad_dict[clients[0]])):
        layer_sum = None
        for c in clients:
            g = grad_dict[c][layer_idx]
            if layer_sum is None:
                layer_sum = g.astype(np.float64)
            else:
                layer_sum += g
        mean_grad.append((layer_sum / num_clients).astype(np.float32))

    mean_sq_norm = grad_sq_norm(mean_grad)

    div_rows = []
    div_vals = []
    div_norm_vals = []

    for c in clients:
        dot_k_mean = grad_dot(grad_dict[c], mean_grad)
        sq_k = sqnorm_dict[c]

        div_sq = max(sq_k + mean_sq_norm - 2.0 * dot_k_mean, 0.0)
        div_norm = math.sqrt(div_sq) / (math.sqrt(mean_sq_norm) + 1e-12)

        div_rows.append({
            "client": c,
            "loss_on_batch": loss_dict[c],
            "div_to_mean_sq": float(div_sq),
            "div_to_mean_norm": float(div_norm),
        })
        div_vals.append(div_sq)
        div_norm_vals.append(div_norm)

    summary = {
        "pairwise": pd.DataFrame(pair_rows),
        "per_client": pd.DataFrame(div_rows),
        "overall_mean_pairwise_grad_l2": float(np.mean(pair_dists)),
        "overall_mean_div_to_mean_sq": float(np.mean(div_vals)),
        "overall_mean_div_to_mean_norm": float(np.mean(div_norm_vals)),
    }
    return summary

In [39]:
# =========================================================
# 主实验
# =========================================================
def run_heterogeneity_experiment(
    base_files,
    feature_cols,
    target_cols,
    geom_cols,
    total_size=None,
    batch_size=64,
    seed=42,
    save_dir="./heterogeneity_results_v2",
):
    os.makedirs(save_dir, exist_ok=True)

    base_dfs = load_base_datasets(base_files)

    # 先保存“原始真实频率对”的 output distance
    base_output_summary = compute_output_distance_for_scenario(
        base_dfs=base_dfs,
        scenario_name="high",   # 这里只是为了拿 raw_pairwise_base
        geom_cols=geom_cols,
        target_cols=target_cols,
    )
    raw_pair_path = os.path.join(save_dir, "raw_base_output_distance_pairwise.csv")
    base_output_summary["raw_pairwise_base"].to_csv(raw_pair_path, index=False)
    print(f"Saved raw base pairwise output distance: {raw_pair_path}")

    scenario_results = []
    final_pairwise_rows = []

    for scenario_name in ["low", "mid", "high"]:
        print("\n" + "=" * 80)
        print(f"Scenario: {scenario_name}")
        print("=" * 80)

        # -------------------------------------------------
        # 1) output distance
        # 用原始三频数据 + 场景混合比例做期望加权
        # -------------------------------------------------
        out_summary = compute_output_distance_for_scenario(
            base_dfs=base_dfs,
            scenario_name=scenario_name,
            geom_cols=geom_cols,
            target_cols=target_cols,
        )

        out_pair_df = out_summary["pairwise"]
        out_pair_path = os.path.join(save_dir, f"{scenario_name}_output_distance_pairwise.csv")
        out_pair_df.to_csv(out_pair_path, index=False)

        print("\n[Output Distance] pairwise:")
        print(out_pair_df)
        print(f"[Output Distance] overall_mean = {out_summary['overall_mean_pairwise_l2']:.6f}")

        # -------------------------------------------------
        # 2) gradient divergence
        # 用该场景下混合后的客户端数据，且不改 freq
        # -------------------------------------------------
        client_dfs = build_scenario_client_datasets(
            base_dfs=base_dfs,
            scenario_name=scenario_name,
            total_size=total_size,
            seed=seed,
        )

        # 把场景客户端数据存下来
        for client_id, df in client_dfs.items():
            save_path = os.path.join(save_dir, f"{scenario_name}_{str(client_id).replace('.', '')}Train.csv")
            df.to_csv(save_path, index=False)
            print(f"Saved client dataset: {save_path} ({len(df)} samples)")

        grad_summary = compute_gradient_divergence_summary(
            client_dfs=client_dfs,
            feature_cols=feature_cols,
            target_cols=target_cols,
            batch_size=batch_size,
            seed=seed,
        )

        grad_pair_df = grad_summary["pairwise"]
        grad_client_df = grad_summary["per_client"]

        grad_pair_path = os.path.join(save_dir, f"{scenario_name}_gradient_pairwise.csv")
        grad_client_path = os.path.join(save_dir, f"{scenario_name}_gradient_per_client.csv")

        grad_pair_df.to_csv(grad_pair_path, index=False)
        grad_client_df.to_csv(grad_client_path, index=False)
        # -------------------------------------------------
        # 3) 汇总成最终论文表：scenario + pair + output + gradient
        # -------------------------------------------------
        merged_pair_df = pd.merge(
            out_pair_df[["scenario", "client_pair", "output_distance"]],
            grad_pair_df[["client_pair", "gradient_distance"]],
            on="client_pair",
            how="inner"
        )

        final_pairwise_rows.append(merged_pair_df)

        print("\n[Gradient Divergence] pairwise:")
        print(grad_pair_df)
        print("\n[Gradient Divergence] per client:")
        print(grad_client_df)
        print(f"[Gradient Distance] overall_mean = {grad_summary['overall_mean_pairwise_grad_l2']:.6f}")

        scenario_results.append({
            "scenario": scenario_name,
            "output_distance": out_summary["overall_mean_pairwise_l2"],
            "gradient_distance": grad_summary["overall_mean_pairwise_grad_l2"],
        })

    summary_df = pd.DataFrame(scenario_results)
    summary_path = os.path.join(save_dir, "heterogeneity_summary.csv")
    summary_df.to_csv(summary_path, index=False)

    # =====================================================
    # 最终论文用 pairwise 表
    # =====================================================
    final_pairwise_table = pd.concat(final_pairwise_rows, ignore_index=True)

    # 排序，保证 low/mid/high + pair 顺序稳定
    scenario_order = {"low": 0, "mid": 1, "high": 2}
    pair_order = {
        "2.4 vs 2.5": 0,
        "2.4 vs 2.6": 1,
        "2.5 vs 2.6": 2,
    }

    final_pairwise_table["scenario_order"] = final_pairwise_table["scenario"].map(scenario_order)
    final_pairwise_table["pair_order"] = final_pairwise_table["client_pair"].map(pair_order)

    final_pairwise_table = final_pairwise_table.sort_values(
        ["scenario_order", "pair_order"]
    ).drop(columns=["scenario_order", "pair_order"]).reset_index(drop=True)

    final_pairwise_path = os.path.join(save_dir, "heterogeneity_pairwise_final_table.csv")
    final_pairwise_table.to_csv(final_pairwise_path, index=False)

    print("\n" + "=" * 80)
    print("Final summary")
    print("=" * 80)
    print(summary_df)
    print(f"\nSaved summary to: {summary_path}")

    print("\n" + "=" * 80)
    print("Final pairwise table for paper")
    print("=" * 80)
    print(final_pairwise_table.to_string(index=False))
    print(f"\nSaved final pairwise table to: {final_pairwise_path}")

    return summary_df

In [40]:
summary_df = run_heterogeneity_experiment(
    base_files=BASE_FILES,
    feature_cols=FEATURE_COLS,
    target_cols=TARGET_COLS,
    geom_cols=GEOM_COLS,
    total_size=None,
    batch_size=BATCH_SIZE,
    seed=SEED,
    save_dir="./heterogeneity_results_v2",
)

Loaded 2.4 GHz: 28500 samples from ./24Train.csv
Loaded 2.5 GHz: 28500 samples from ./25Train.csv
Loaded 2.6 GHz: 28500 samples from ./26Train.csv
Saved raw base pairwise output distance: ./heterogeneity_results_v2/raw_base_output_distance_pairwise.csv

Scenario: low

[Output Distance] pairwise:
  scenario client_pair  client_a  client_b  output_distance  \
0      low  2.4 vs 2.5       2.4       2.5         0.351514   
1      low  2.4 vs 2.6       2.4       2.6         0.351514   
2      low  2.5 vs 2.6       2.5       2.6         0.351514   

   output_distance_std  
0             0.280035  
1             0.280035  
2             0.280035  
[Output Distance] overall_mean = 0.351514
Saved client dataset: ./heterogeneity_results_v2/low_24Train.csv (28500 samples)
Saved client dataset: ./heterogeneity_results_v2/low_25Train.csv (28500 samples)
Saved client dataset: ./heterogeneity_results_v2/low_26Train.csv (28500 samples)

[Gradient Divergence] pairwise:
  client_pair  client_a  client_